<a href="https://colab.research.google.com/github/SindhuBangaru/ai-mentor-portfolio/blob/main/Day10_MultiAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "numpy<2.0"
!pip install -q --upgrade crewai litellm google-generativeai google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 62.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xa

In [ ]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

Enter Gemini API Key: ··········
API key set


In [ ]:
from crewai import Agent, Task, Crew, Process

print("CrewAI imported successfully")

ModuleNotFoundError: No module named 'crewai'

In [ ]:
llm = "gemini/gemini-2.5-flash"

print(llm)

In [ ]:
researcher = Agent(
    role="Placement Researcher",
    goal="Compile concise, factual placement-prep notes for B.Tech students",
    backstory="You are a research analyst who prepares short factual placement briefs in bullet form.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("Researcher agent created")

In [ ]:
writer = Agent(
    role="Placement Brief Writer",
    goal="Convert research notes into a clean 1-page placement brief in markdown",
    backstory="You write clear student-friendly placement briefs with headings and actionable takeaways.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("Writer agent created")

In [ ]:
research_task = Task(
    description=(
        "Research TCS Digital placement preparation for B.Tech students. "
        "Include hiring process, technical stack, eligibility, recent hiring trends, "
        "and preparation tips. Give 3-5 bullets per section."
    ),
    agent=researcher,
    expected_output=(
        "Markdown bullet list with 5 sections: Hiring Process, Technical Stack, "
        "Eligibility, Recent Hiring Trends, Prep Tips."
    ),
)

print("Research task created")

In [ ]:
write_task = Task(
    description=(
        "Using the research notes, produce a 1-page placement brief for B.Tech students. "
        "Use markdown format. Include a 3-line opening hook, 5 clear sections, "
        "and a 3-line closing call-to-action."
    ),
    agent=writer,
    expected_output='1-page markdown brief titled "TCS Digital — Placement Prep Brief".',
)

print("Writer task created")

In [ ]:
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)

print("Crew created successfully")

In [ ]:
result = await crew.kickoff_async()

print("\n\n=== FINAL OUTPUT ===\n")
print(result)

In [ ]:
import pathlib

pathlib.Path("day10_lab10a_transcript.txt").write_text(str(result))

print("Saved: day10_lab10a_transcript.txt")
print("Characters:", len(str(result)))

In [ ]:
try:
    for i, output in enumerate(result.tasks_output):
        print(f"\n=== Task {i+1} Output ===")
        print(str(output)[:800])
except Exception:
    print("Task outputs not available in this CrewAI version.")
    print("Final result saved successfully.")

In [ ]:
markdown_content = str(result)

with open("tcs_digital_brief.md", "w", encoding="utf-8") as f:
    f.write(markdown_content)

print("Markdown file created successfully: tcs_digital_brief.md")

In [ ]:
from google.colab import files

files.download("tcs_digital_brief.md")

In [ ]:
with open("tcs_digital_brief.md", "r", encoding="utf-8") as f:
    print(f.read()[:2000])

In [ ]:
llm = "gemini/gemini-2.5-flash"

'''crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)'''

result = await crew.kickoff_async()

In [ ]:
import json
import pathlib

In [ ]:
!pip install -q "numpy<2.0"
!pip install -q --upgrade crewai litellm google-generativeai google-genai chromadb sentence-transformers crewai-tools

In [ ]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool as crewai_tool

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

import json
import pathlib

In [ ]:
llm = "gemini/gemini-2.5-flash"

print(llm)

In [ ]:
client_db = PersistentClient(path='./chroma_db')

col = client_db.get_or_create_collection('placement_kb')

embed = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

@crewai_tool
def rag_search(query: str) -> str:
    """
    Search placement knowledge base.
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results['documents'][0]

    return '\n---\n'.join(docs)

In [ ]:
researcher = Agent(
    role="Placement Researcher",

    goal="Research company placement requirements for students",

    backstory="You prepare factual placement research notes using RAG search.",

    llm=llm,

    tools=[rag_search],

    verbose=True,

    allow_delegation=False,
)

In [ ]:
interviewer = Agent(
    role="Mock Interviewer",

    goal="Generate placement interview questions",

    backstory="You generate technical and HR interview questions.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [ ]:
coach = Agent(
    role="Answer Coach",

    goal="Create strong sample answers and preparation guidance",

    backstory="You coach students with strong placement answers.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [ ]:
tracker = Agent(
    role="Progress Tracker",

    goal="Create structured student progress summaries",

    backstory="You generate JSON-style student progress summaries.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [ ]:
profiles = [
    {
        "student_id": "S001",
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 7.8,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS Digital"
    },

    {
        "student_id": "S002",
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.1,
        "skills": ["Python", "DBMS"],
        "target_company": "Cognizant"
    },

    {
        "student_id": "S003",
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.5,
        "skills": ["Java", "DSA", "OOPs"],
        "target_company": "Amazon"
    }
]

In [ ]:
def build_tasks(profile):

    research = Task(
        description=(
            f"Research placement preparation details for "
            f"{profile['target_company']} "
            f"for a {profile['branch']} student."
        ),

        agent=researcher,

        expected_output="Short bullet research notes."
    )

    interview = Task(
        description=(
            f"Generate EXACTLY 10 mock interview questions "
            f"for {profile['name']} targeting "
            f"{profile['target_company']}."
        ),

        agent=interviewer,

        expected_output="Exactly 10 numbered interview questions."
    )

    coaching = Task(
        description=(
            f"Pick question 3 and create strong sample answer "
            f"for {profile['name']}."
        ),

        agent=coach,

        expected_output="One strong answer and 3 preparation tips."
    )

    tracking = Task(
        description=(
            f"Create JSON-style progress summary "
            f"for {profile['student_id']}."
        ),

        agent=tracker,

        expected_output="Valid JSON-style summary."
    )

    return [research, interview, coaching, tracking]

In [ ]:
p = profiles[0]

crew = Crew(
    agents=[
        researcher,
        interviewer,
        coach,
        tracker
    ],

    tasks=build_tasks(p),

    process=Process.sequential,

    verbose=True,
)

In [ ]:
result = await crew.kickoff_async()

print("\n=== FINAL OUTPUT ===\n")

print(result)

In [ ]:
transcripts = []

for p in profiles:

    print("\n" + "="*60)

    print(f"Running for {p['name']} → {p['target_company']}")

    print("="*60)

    crew = Crew(
        agents=[
            researcher,
            interviewer,
            coach,
            tracker
        ],

        tasks=build_tasks(p),

        process=Process.sequential,

        verbose=True,
    )

    result = await crew.kickoff_async()

    transcripts.append({
        "student": p["name"],
        "target": p["target_company"],
        "final_output": str(result),
    })

print("Completed:", len(transcripts))

In [ ]:
pathlib.Path(
    "day10_sprint5_transcripts.json"
).write_text(
    json.dumps(transcripts, indent=2)
)

print("Saved transcripts successfully")

In [ ]:
md = "# Day10 Sprint5 Report\n\n"

for t in transcripts:

    md += f"## {t['student']} → {t['target']}\n\n"

    md += t["final_output"]

    md += "\n\n---\n\n"

pathlib.Path(
    "day10_sprint5_report.md"
).write_text(md)

print("Markdown report created")

In [ ]:
from google.colab import files

files.download("day10_sprint5_transcripts.json")

files.download("day10_sprint5_report.md")